In [1]:
import os
os.chdir("../..")

In [2]:
import torch
from models.mlrod import MLROD_model
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
class Pharma_dataset(Dataset):
  def __init__(self,path):
    """
    path is a string containing the path to the pkl dataset
    """
    super().__init__()   
    #X is a list with each element of the list containing a 1024 time series data 
    #y is a list with containing the name of the chemical corresponding to X
    self.y, self.X = pickle.load(open(path, 'rb'))
    names = sorted(list(set(self.y)))
    self.mapping = {names[i]:i for i in range(len(names))}   #Maps each material name to a number

  def __len__(self):
    return len(self.y)

  def __getitem__(self,index):
    data = torch.Tensor(self.X[index]) #of shape (1,1024)
    data = data/data.max()
    label = self.mapping[self.y[index]]
    return data,label

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
test_set = Pharma_dataset("datasets/Pharma/Pharma_test.pkl")
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

The number of elements in the test set is 704


In [8]:
Pharma_mlrod = MLROD_model(num_classes=32).to(device)
Pharma_mlrod.load_state_dict(torch.load("results/trained_models/Pharma_mlrod_19_97.87_.pt"))

Pharma_RamanFormer = RamanFormer(num_classes=32).to(device)
Pharma_RamanFormer.load_state_dict(torch.load("results/trained_models/Pharma_RamanFormer_28_97.3_.pt"))

Pharma_RamanNet = RamanNet(num_classes=32).to(device)
Pharma_RamanNet.load_state_dict(torch.load("results/trained_models/Pharma_RamanNet_4_97.59_.pt"))

Pharma_SANet = ScaleAdaptiveNet(num_classes=32).to(device)
Pharma_SANet.load_state_dict(torch.load("results/trained_models/Pharma_SANet_13_99.72_.pt"))

Pharma_transformer = ViT(patch_size=128,p=0.1,num_classes=32).to(device)
Pharma_transformer.load_state_dict(torch.load("results/trained_models/Pharma_transformer_27_97.16_.pt"))

<All keys matched successfully>

In [9]:
all_acc, _, _, all_f1 = test_f1(Pharma_mlrod,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_17908/1606997643.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 22/22 [00:01<00:00, 17.43it/s]

96.59\% & 0.9658 \\


In [10]:
all_acc, _, _, all_f1 = test_f1(Pharma_SANet,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_17908/2605796513.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 22/22 [00:01<00:00, 20.28it/s]

99.57\% & 0.9957 \\


In [11]:
all_acc, _, _, all_f1 = test_f1_RamanNet(Pharma_RamanNet,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_17908/1275098365.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 22/22 [00:01<00:00, 20.56it/s]

97.02\% & 0.9662 \\


In [12]:
all_acc, _, _, all_f1 = test_f1(Pharma_transformer,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_17908/1180652513.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 22/22 [00:01<00:00, 17.32it/s]

95.74\% & 0.9492 \\


In [13]:
all_acc, _, _, all_f1 = test_f1(Pharma_RamanFormer,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_17908/1783686368.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 22/22 [00:01<00:00, 21.72it/s]

97.3\% & 0.9683 \\
